# Project: Audio Transcription and Automated Meeting Minutes

I'm building a pipeline to process audio files from city council meetings. The goal is to transcribe the audio and then use a Large Language Model (LLM) to extract structured meeting minutes, including action items.

In [ ]:
# Install necessary libraries for LLM inference and speech recognition
# bitsandbytes & accelerate allow for efficient model loading (quantization)
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# Core imports for file handling, API interaction, and Transformer models
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [ ]:
# Constants
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# Mount Google Drive to access the audio dataset stored in my cloud storage
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/Colab Notebooks/AI Engineer Core Track/week-3/denver_extract.mp3"

In [ ]:
# Authenticate with Hugging Face to download restricted models like Llama
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Prepare the audio file object for transcription APIs
audio_file = open(audio_filename, "rb")

## Step 1: Speech-to-Text (Transcription)
I'll compare two methods: a local Open Source model via Hugging Face and an API-based approach via OpenAI.

### Method A: Local Inference with OpenAI Whisper
Using the `transformers` pipeline to run Whisper locally on the GPU.

In [ ]:
from transformers import pipeline

# Initialize the ASR pipeline with the medium English Whisper model
# I'm using float16 and CUDA to speed up inference on the Colab GPU
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

In [ ]:
open_source_transcription = transcription

### Method B: OpenAI Whisper API
Testing the managed API version for comparison.

In [ ]:
# Sign in to OpenAI using Secrets in Colab
AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

## Step 2: Information Extraction with LLMs
Now I'll pass the transcript to a Llama-3 model to summarize and extract action items.

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

In [ ]:
# Memory Optimization: Configure 4-bit NormalFloat (NF4) quantization
# This allows me to run larger models on consumer-grade/Colab GPUs
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Load Tokenizer and the Model with quantization
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

# Apply the chat template specific to Llama-3 to format system/user roles correctly
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# TextStreamer allows me to see the model generating text in real-time
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

# Generate the response
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))